# Indices

A field does not carry raw Spenso slots. It carries a tuple of `IndexType`
objects. Each `IndexType` wraps a Spenso `Representation` together with a
`kind` string used throughout lowering and the vertex engine.

The usual constants (`SPINOR_INDEX`, `LORENTZ_INDEX`, `COLOR_FUND_INDEX`, ...)
are enough for Standard Model work. This notebook shows what they are, and
that a completely new family can be declared the same way.


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from symbolica.community.spenso import Representation, TensorName

from feynpy import (
    DC,
    FS,
    Field,
    Gamma,
    GaugeGroup,
    GaugeRepresentation,
    IndexType,
    Model,
    Parameter,
)
from symbolic.vertex_engine import I


## From Spenso representations to `IndexType`

The raw objects are Spenso representations such as `bis(4)` and `mink(4)`.
`IndexType` is the project metadata wrapped around one of them.


In [2]:
SPINOR_INDEX = IndexType("Spinor", Representation.bis(4), "spinor", prefix="i")
LORENTZ_INDEX = IndexType("Lorentz", Representation.mink(4), "lorentz", prefix="mu")
COLOR_FUND_INDEX = IndexType("ColorFund", Representation.cof(3), "color_fund", prefix="c")
COLOR_ADJ_INDEX = IndexType("ColorAdj", Representation.coad(8), "color_adj", prefix="a")
WEAK_FUND_INDEX = IndexType("WeakFund", Representation.cof(2), "weak_fund", prefix="w")
WEAK_ADJ_INDEX = IndexType("WeakAdj", Representation.coad(3), "weak_adj", prefix="aw")
GENERATION_INDEX = IndexType(
    "Generation",
    Representation.cof(3),
    "generation",
    dimension=3,
    is_flavor=True,
    prefix="f",
)

for index in (
    SPINOR_INDEX, LORENTZ_INDEX, COLOR_FUND_INDEX, COLOR_ADJ_INDEX,
    WEAK_FUND_INDEX, WEAK_ADJ_INDEX, GENERATION_INDEX,
):
    show(
        index.name,
        f"kind={index.kind}, flavor={index.is_flavor}, rep={index.representation}",
    )


Spinor
kind=spinor, flavor=False, rep=bis(4)

Lorentz
kind=lorentz, flavor=False, rep=mink(4)

ColorFund
kind=color_fund, flavor=False, rep=cof(3)

ColorAdj
kind=color_adj, flavor=False, rep=coad(8)

WeakFund
kind=weak_fund, flavor=False, rep=cof(2)

WeakAdj
kind=weak_adj, flavor=False, rep=coad(3)

Generation
kind=generation, flavor=True, rep=cof(3)



## Fields store `IndexType`, labels come later

A gluon carries Lorentz plus color adjoint. A quark class carries generation,
color, and spinor. Concrete labels are attached only when the field is used.


In [3]:
Gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
QClass = Field(
    "Q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("Q"),
    conjugate_symbol=S("Qbar"),
    indices=(GENERATION_INDEX, COLOR_FUND_INDEX, SPINOR_INDEX),
    flavor_index=GENERATION_INDEX,
    class_members=("u", "c", "t"),
)
RepeatedColor = Field(
    "X",
    spin=0,
    self_conjugate=False,
    symbol=S("X"),
    conjugate_symbol=S("Xdag"),
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)

show("G indices", Gluon.indices)
show("Q indices", QClass.indices)
show("Q members", [member.name for member in QClass.class_members])

packed = QClass.pack_slot_labels({0: S("f"), 1: S("c"), 2: S("alpha")})
show("kind-keyed Q labels", packed)
show("unpacked slot labels", QClass.unpack_slot_labels(packed))
show("repeated color labels", RepeatedColor.pack_slot_labels({0: S("c1"), 1: S("c2")}))


G indices
(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=None, is_flavor=False, prefix='a'))

Q indices
(IndexType(name='Generation', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='generation', dimension=3, is_flavor=True, prefix='f'), IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', dimension=None, is_flavor=False, prefix='c'), IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', dimension=None, is_flavor=False, prefix='i'))

Q members
['u', 'c', 't']

kind-keyed Q labels
3 vertex signature(s)

Vertex: generation
Rule: f

Vertex: color_fund
Rule: c

Vertex: spinor
Rule: alpha

unpacked slot l

Flavor is not a separate engine at declaration time. It is an `IndexType`
with `is_flavor=True`. Expansion happens later, at `feynman_rule(..., flavor_expand=True)`.


In [4]:
Yu = Parameter("Yu", indices=(GENERATION_INDEX, GENERATION_INDEX))
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))
f, h, c, alpha = S("f"), S("h"), S("c"), S("alpha")

yukawa = Model(
    Yu(f, h)
    * QClass.bar(index_labels={"generation": f, "color_fund": c, "spinor": alpha})
    * QClass(index_labels={"generation": h, "color_fund": c, "spinor": alpha})
    * Phi,
    parameters=(Yu,),
)
show_model(yukawa, QClass.bar, QClass, Phi)


Lagrangian
Yu(f,h) * Q.bar * Q * Phi

Feynman Rule
1𝑖*g(bis(4, i1),bis(4, i2))*g(cof(3, c1),cof(3, c2))*Yu(f1,f2)



## Gauge generators follow the declared indices

The same explicit `IndexType` objects control which generator tensor `DC`
inserts.


In [5]:
from symbolic.spenso_structures import gauge_generator, structure_constant, weak_gauge_generator, weak_structure_constant

WBoson = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)
LClass = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L"),
    conjugate_symbol=S("Lbar"),
    indices=(GENERATION_INDEX, WEAK_FUND_INDEX, SPINOR_INDEX),
    flavor_index=GENERATION_INDEX,
    class_members=("e", "mu", "ta"),
)

SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=S("gS"),
    gauge_boson=Gluon,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fundamental"),
    ),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=S("g2"),
    gauge_boson=WBoson,
    structure_constant=weak_structure_constant,
    representations=(
        GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet"),
    ),
)

mu = S("mu")
qcd_model = Model(
    I * QClass.bar * Gamma(mu) * DC(QClass, mu),
    gauge_groups=(SU3C,),
)
weak_model = Model(
    I * LClass.bar * Gamma(mu) * DC(LClass, mu),
    gauge_groups=(SU2L,),
)
show_model(qcd_model, QClass.bar, QClass, Gluon)
show_model(weak_model, LClass.bar, LClass, WBoson)


Lagrangian
1𝑖 * Q.bar * Gamma(mu) * DC(Q, mu)

Feynman Rule
1𝑖*gS*g(cof(3, f1),cof(3, f2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

Lagrangian
1𝑖 * L.bar * Gamma(mu) * DC(L, mu)

Feynman Rule
1𝑖*g2*g(cof(3, f1),cof(3, f2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))



## A custom index family

Nothing below is imported from the built-in colour or weak helpers. A
5-dimensional matter index and a 24-dimensional adjoint index are enough to
run both `DC` and `FS`.


In [6]:
RIBBON_FUND = IndexType("RibbonFund", Representation.cof(5), "ribbon_fund", prefix="rb")
RIBBON_ADJ = IndexType("RibbonAdj", Representation.coad(24), "ribbon_adj", prefix="ra")


def ribbon_generator(adj, left, right):
    return TensorName.t()(
        Representation.coad(24)(adj),
        Representation.cof(5)(left),
        Representation.cof(5)(right),
    ).to_expression()


def ribbon_structure(a, b, c):
    return TensorName.f()(
        Representation.coad(24)(a),
        Representation.coad(24)(b),
        Representation.coad(24)(c),
    ).to_expression()


R = Field("R", spin=1, self_conjugate=True, symbol=S("R"), indices=(LORENTZ_INDEX, RIBBON_ADJ))
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("Xi"),
    conjugate_symbol=S("Xibar"),
    indices=(RIBBON_FUND, SPINOR_INDEX),
)
Ribbon = GaugeGroup(
    name="RibbonSU5",
    abelian=False,
    coupling=S("gM"),
    gauge_boson=R,
    structure_constant=ribbon_structure,
    representations=(
        GaugeRepresentation(index=RIBBON_FUND, generator_builder=ribbon_generator, name="twisted_five"),
    ),
)
nu, aa = S("nu"), S("a")
ribbon_model = Model(
    I * Xi.bar * Gamma(mu) * DC(Xi, mu)
    - (Expression.num(1) / Expression.num(4)) * FS(Ribbon, mu, nu, aa) * FS(Ribbon, mu, nu, aa),
    gauge_groups=(Ribbon,),
)
show_model(ribbon_model, Xi.bar, Xi, R)
show("three-boson vertex from FS", ribbon_model.feynman_rule(R, R, R, include_delta=False))


Lagrangian
1𝑖 * Xi.bar * Gamma(mu) * DC(Xi, mu) + -1/4 * FS(RibbonSU5, mu, nu, a) * FS(RibbonSU5, mu, nu, a)

Feynman Rule
1𝑖*gM*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(24, ra3),cof(5, rb1),cof(5, rb2))

three-boson vertex from FS
-gM*g(mink(4, mu3),mink(4, mu1))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q1,mu2)+gM*g(mink(4, mu3),mink(4, mu1))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q3,mu2)+gM*g(mink(4, mu3),mink(4, mu2))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q2,mu1)-gM*g(mink(4, mu3),mink(4, mu2))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q3,mu1)+gM*g(mink(4, mu1),mink(4, mu2))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q1,mu3)-gM*g(mink(4, mu1),mink(4, mu2))*f(coad(24, ra1),coad(24, ra2),coad(24, ra3))*pcomp(q2,mu3)

